# EDA y Regresión — Análisis Nutricional y Densidad Calórica
**Lead University · Minería de Datos · Tarea 2**

Exploración visual, ANOVA y regresión lineal múltiple sobre el dataset `daily_food_nutrition_dataset.csv`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from statsmodels.nonparametric.smoothers_lowess import lowess

from scripts import (
    GraficosCuantitativos,
    TestEstadisticos,
    RegresionLineal,
)

sns.set(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 6)
print('Librerías cargadas.')

In [ ]:
df_raw = pd.read_csv('datos/daily_food_nutrition_dataset.csv')

# Renombrar columnas a nombres simples para facilitar el trabajo
df = df_raw.rename(columns={
    'Food_Item':          'food',
    'Category':           'category',
    'Calories (kcal)':    'calories',
    'Protein (g)':        'protein',
    'Carbohydrates (g)':  'carbs',
    'Fat (g)':            'fat',
    'Fiber (g)':          'fiber',
    'Sugars (g)':         'sugar',
    'Sodium (mg)':        'sodium',
    'Cholesterol (mg)':   'cholesterol',
    'Meal_Type':          'meal_type',
    'Water_Intake (ml)':  'water',
})

print(f'Registros: {df.shape[0]:,}  |  Variables: {df.shape[1]}')
print(f'Categorías: {sorted(df["category"].unique())}')
df.head()

---
## Fase 1 — Minería y Perfilamiento

### 1.1 Composición por Categoría — Barras Apiladas de Macronutrientes

Promedio de `protein`, `carbs` y `fat` para las **10 categorías** con más registros.

In [ ]:
top10 = df['category'].value_counts().head(10).index.tolist()
df_top = df[df['category'].isin(top10)]

macro_avg = (
    df_top.groupby('category')[['protein', 'carbs', 'fat']]
    .mean()
    .loc[top10]  # mantener orden por frecuencia
    .rename(columns={'protein': 'Proteína (g)', 'carbs': 'Carbohidratos (g)', 'fat': 'Grasa (g)'})
)

fig, ax = plt.subplots(figsize=(13, 6))
macro_avg.plot(kind='bar', stacked=True, ax=ax, colormap='Set2')
ax.set_title('Promedio de macronutrientes por categoría (Top 10 con más registros)', fontsize=13)
ax.set_xlabel('Categoría')
ax.set_ylabel('Gramos promedio por porción')
ax.legend(title='Macronutriente', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

print('Registros por categoría (Top 10):')
print(df['category'].value_counts().head(10).to_string())

**Interpretación:**

- Las categorías proteicas (Protein/Dairy, Meat/Poultry) dominan en proteína por porción, siendo las fuentes más eficientes para dietas de alto rendimiento atlético o recuperación muscular.
- Grains y Snacks presentan el mayor aporte de carbohidratos, siendo fuentes calóricas densas con mayor índice glucémico, relevante para pacientes con diabetes.
- La grasa sobresale en Fats/Oils. Dado que 1 g de grasa ≈ 9 kcal (vs 4 kcal/g de proteína y carbs), estas categorías concentran el mayor riesgo de exceso calórico por volumen pequeño.
- Este gráfico es una herramienta de **perfilamiento nutricional rápido** útil para dietistas al diseñar menús balanceados o auditar la composición de dietas existentes.

### 1.2 Relación Calórica — Scatter `fat` vs `calories` con curva LOWESS

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(df['fat'], df['calories'], alpha=0.3, s=15, color='steelblue', label='Alimentos')

# Curva LOWESS sobre los datos ordenados
xy = df[['fat', 'calories']].dropna().sort_values('fat')
smooth = lowess(xy['calories'].values, xy['fat'].values, frac=0.3)
ax.plot(smooth[:, 0], smooth[:, 1], 'r-', lw=2.5, label='Tendencia LOWESS')

ax.set_xlabel('Grasa (g)')
ax.set_ylabel('Calorías (kcal)')
ax.set_title('Relación Calorías ~ Grasa con tendencia LOWESS')
ax.legend()
plt.tight_layout()
plt.show()

**Interpretación:**

- La curva LOWESS revela una tendencia **positiva y aproximadamente lineal**: a mayor contenido de grasa, mayor densidad calórica, esperado dado que 1 g grasa = 9 kcal.
- La **alta dispersión** alrededor de la curva indica que alimentos con igual contenido de grasa pueden diferir mucho en calorías totales según su contenido de carbohidratos y proteínas.
- Los puntos con altas calorías y poca grasa (zona superior izquierda) corresponden típicamente a alimentos muy ricos en azúcares o almidones refinados.
- La ventaja de LOWESS frente a una regresión lineal simple es que captura curvatura local sin asumir linealidad global, revelando si la relación grasa-caloría se aplana o acelera en rangos extremos.

---
## Fase 2 — ANOVA y Comparaciones Múltiples

¿Varía significativamente el contenido de `sugar` entre **Fruit** y **Grain**?

- **H₀:** La media de azúcar es igual en ambas categorías.
- **H₁:** Las medias difieren (α = 0.05).

In [ ]:
df_fg = df[df['category'].isin(['Fruit', 'Grain'])].copy()

print('Registros por categoría:')
print(df_fg['category'].value_counts().to_string())
print()
print(df_fg.groupby('category')['sugar'].agg(['mean', 'median', 'std']).round(2))

In [ ]:
tests_nut = TestEstadisticos(df_fg)

print('=== ANOVA de una vía: sugar ~ category ===')
display(tests_nut.anova('sugar', 'category'))

res_f = tests_nut.anova_scipy('sugar', 'category')
print(f"F = {res_f['F']:.4f},  p-valor = {res_f['p_valor']:.6f}")
print('Decisión:', 'Rechazar H₀ — diferencia significativa' if res_f['p_valor'] < 0.05 else 'No rechazar H₀')

### 2.2 Post-Hoc — Distribución del azúcar por categoría

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

g_fg = GraficosCuantitativos(df_fg)
g_fg.violin(x='category', y='sugar', ax=axes[0])
g_fg.boxplot(x='category', y='sugar', ax=axes[1])

plt.suptitle('Distribución del Azúcar: Fruit vs Grain', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

**Interpretación:**

- Si p < 0.05 en el ANOVA existe evidencia de que el contenido medio de azúcar difiere entre frutas y granos.
- Las **frutas** presentan mayor variabilidad: desde frutas con poca azúcar (aguacate, limón) hasta frutas tropicales muy dulces (mango, plátano), lo que genera la cola superior visible en el violin.
- Los **granos** muestran distribuciones más compactas y asimétricas hacia la derecha: el azúcar en granos proviene del almidón hidrolizado y es generalmente menor en frutas como azúcar libre.
- **Relevancia clínica:** Esta diferencia es fundamental en planes nutricionales para diabetes tipo 2 o síndrome metabólico, donde controlar el azúcar libre es prioritario.

---
## Fase 3 — Regresión Lineal Múltiple

**Modelo:**

$$\text{Calories} = \beta_0 + \beta_1(\text{protein}) + \beta_2(\text{carbs}) + \beta_3(\text{fat}) + \beta_4(\text{sugar}) + \varepsilon$$

In [ ]:
df_reg = df[['calories', 'protein', 'carbs', 'fat', 'sugar']].dropna()
print(f'Registros para la regresión: {len(df_reg):,}')
print(df_reg.describe().round(2))

In [ ]:
reg = RegresionLineal(
    'calories ~ protein + carbs + fat + sugar',
    df_reg,
).ajustar()

print(reg.resumen())

In [ ]:
print('=== Coeficientes del modelo ===')
display(reg.coeficientes())

print('\n=== Bondad de ajuste ===')
display(reg.bondad_ajuste())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
reg.grafico_predichos_vs_observados(ax=axes[0])
reg.grafico_regresion('fat', 'calories', ax=axes[1])
plt.tight_layout()
plt.show()

reg.diagnostico_residuos()
plt.show()

**Interpretación de coeficientes:**

| Predictor | Valor teórico (Atwater) | Interpretación |
|-----------|------------------------|----------------|
| `protein` | 4 kcal/g | Cada gramo de proteína aporta ~4 kcal. Coeficiente esperado. |
| `carbs`   | 4 kcal/g | Cada gramo de carbohidrato aporta ~4 kcal. |
| `fat`     | 9 kcal/g | La grasa tiene la mayor densidad energética (~9 kcal/g), el coeficiente más alto del modelo. |
| `sugar`   | subconjunto de carbs | Puede ser no significativo (p > 0.05) o reducido porque el azúcar ya está capturado en `carbs` (colinealidad). |

**Interpretación del R² ajustado:**

El R² ajustado mide la **proporción de varianza en calorías explicada por la composición química**, penalizando por el número de predictores para evitar sobreajuste.

- Un R² ajustado **alto (> 0.95)** confirma que la composición química determina casi completamente las calorías, coherente con el sistema de Atwater que define las calorías precisamente a partir de macronutrientes.
- Si R² ajustado fuera bajo, indicaría que otros componentes no incluidos (alcohol, fibra soluble fermentable, densidad energética del agua) influyen en el valor calórico real.
- La diferencia entre R² y R² ajustado es mínima cuando todos los predictores son relevantes — si `sugar` no aporta (p > 0.05), el R² ajustado lo penalizará.

---
## Conclusiones

El análisis nutricional revela patrones robustos y clínicamente interpretables:

1. **Perfilamiento por categoría:** Las barras apiladas evidencian diferencias macronutricionales claras entre grupos de alimentos, herramienta clave para diseñar dietas balanceadas.
2. **Relación grasa-calorías:** La curva LOWESS confirma una asociación positiva con dispersión significativa, explicada por la variedad en la composición de macronutrientes.
3. **ANOVA azúcar:** Se detecta diferencia estadísticamente significativa en el contenido de azúcar entre frutas y granos, con relevancia directa en nutrición clínica.
4. **Regresión lineal:** El modelo basado en macronutrientes explica una proporción alta de la varianza calórica, validando el modelo de Atwater y confirmando que proteína, carbohidratos y grasa son los determinantes primarios del valor energético de los alimentos.